# Milestone 2: Data Cleaning dan Quality Profiling — Trase Indonesia Palm Oil

Notebook revisi ini mempertahankan seluruh tahapan Milestone 2 dengan pendekatan lebih hemat memori:
1. Membaca dan memprofilkan CSV besar menggunakan DuckDB `SUMMARIZE`.
2. Memeriksa missing values, termasuk string kosong/spasi.
3. Menangani missing values pada kolom ukuran numerik dan kolom teks.
4. Mendeteksi serta menangani outlier menggunakan IQR/Winsorization pada satu kolom ukuran yang relevan.
5. Menghapus baris yang identik di seluruh kolom (tidak menganggap ID pabrik sebagai kunci unik).
6. Memvalidasi jumlah baris, null, dan rentang tahun.
7. Menyimpan hasil Parquet dan laporan cleaning.

**Optimasi memori:** CSV diproses dengan Polars LazyFrame; tidak dibuat banyak salinan DataFrame di RAM. Deduplikasi dilakukan oleh DuckDB dengan batas memori dan direktori sementara agar bisa menggunakan disk bila diperlukan.

Jalankan notebook dari root repository atau folder `notebooks`. CSV asli harus berada di `data/raw/indonesia_palm_oil_v1_2_5.csv`.

In [1]:
from pathlib import Path
import polars as pl
import duckdb

# Tentukan root proyek
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "indonesia_palm_oil_v1_2_5.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = OUTPUT_DIR / "indonesia_palm_oil_bersih.parquet"
TEMP_PATH = OUTPUT_DIR / "indonesia_palm_oil_sementara.parquet"
REPORT_PATH = OUTPUT_DIR / "ringkasan_cleaning.txt"
DUCKDB_TEMP_DIR = OUTPUT_DIR / "duckdb_tmp"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATA_PATH}")

print("Folder proyek :", PROJECT_ROOT)
print("Dataset       :", DATA_PATH)
print(f"Ukuran CSV    : {DATA_PATH.stat().st_size / (1024**2):,.2f} MB")
print("Polars        :", pl.__version__)
print("DuckDB        :", duckdb.__version__)

# LazyFrame: data belum dimuat seluruhnya ke RAM.
lf = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
    null_values=["", "NA", "N/A", "NULL", "null", "None"],
    low_memory=True,
)
schema_awal = lf.collect_schema()
kolom_awal = schema_awal.names()
print("Jumlah kolom:", len(kolom_awal))
print("Nama kolom:", kolom_awal)

# DuckDB dikonfigurasi agar pemrosesan besar dapat menggunakan disk sementara.
con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute("SET threads=2")
con.execute(f"SET temp_directory='{str(DUCKDB_TEMP_DIR).replace(chr(39), chr(39)*2)}'")
con.execute("SET preserve_insertion_order=false")

# Escape path untuk literal SQL DuckDB.
def sql_path(path):
    return str(path).replace("'", "''")

print("Persiapan selesai. Data akan diproses secara lazy.")

Folder proyek : /home/jovyan/work
Dataset       : /home/jovyan/work/data/raw/indonesia_palm_oil_v1_2_5.csv
Ukuran CSV    : 721.83 MB
Polars        : 2.0.0
DuckDB        : 2.0.0
Jumlah kolom: 34
Nama kolom: ['year', 'country_of_production', 'product_type', 'forest_500_palm_oil', 'zero_deforestation_indonesia_palm_oil', 'province_of_production', 'kabupaten_of_production', 'mill', 'mill_trase_id', 'mill_group', 'refinery', 'refinery_trase_id', 'refinery_group', 'exporter', 'exporter_group', 'port_of_export', 'importer', 'importer_group', 'country_of_first_import', 'economic_bloc', 'province_of_production_trase_id', 'kabupaten_of_production_trase_id', 'country_of_production_trase_id', 'country_of_first_import_trase_id', 'palm_oil_deforestation_10_year_total_exposure', 'palm_oil_deforestation_annual_exposure', 'total_emission_exposure', 'net_emission_luc_exposure', 'emission_subsidence_exposure', 'gross_emission_luc_exposure', 'emissions_fire_on_peat_exposure', 'volume', 'oil_palm_ha', 'fob

## 1. Profiling awal dengan DuckDB `SUMMARIZE`

DuckDB membaca CSV langsung dari disk dan mengembalikan tabel ringkasan yang relatif kecil. Ini menghindari konversi seluruh dataset menjadi Pandas hanya untuk profiling.

In [2]:
profil_awal = con.execute(
    f"SUMMARIZE SELECT * FROM read_csv_auto('{sql_path(DATA_PATH)}', sample_size=10000, ignore_errors=false)"
).fetchdf()
display(profil_awal)

# Jumlah baris awal dihitung oleh DuckDB tanpa memuat semua baris ke Python.
total_baris_awal = con.execute(
    f"SELECT COUNT(*) FROM read_csv_auto('{sql_path(DATA_PATH)}', sample_size=10000, ignore_errors=false)"
).fetchone()[0]
print(f"Jumlah baris awal: {total_baris_awal:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,year,BIGINT,2013,2022,10,2020.0519167128343,1.3540499690553924,2019,2020,2021,1694907,0.00
1,country_of_production,VARCHAR,INDONESIA,INDONESIA,1,NaN,NaN,NaN,NaN,NaN,1694907,0.00
2,product_type,VARCHAR,PALM OIL,REFINED PALM OIL,2,NaN,NaN,NaN,NaN,NaN,1694907,0.00
3,forest_500_palm_oil,VARCHAR,0,NOT ASSESSED,7,NaN,NaN,NaN,NaN,NaN,1694907,0.00
4,zero_deforestation_indonesia_palm_oil,VARCHAR,COMPANY COMMITMENT,UNKNOWN,4,NaN,NaN,NaN,NaN,NaN,1694907,0.00
5,province_of_production,VARCHAR,ACEH,UNKNOWN,30,NaN,NaN,NaN,NaN,NaN,1694907,0.00
6,kabupaten_of_production,VARCHAR,ACEH BARAT,WAY KANAN,282,NaN,NaN,NaN,NaN,NaN,1694907,0.00
7,mill,VARCHAR,ABDI BORNEO PLANTATIONS (ABDI),YUNI BERSAUDARA SEJAHTERA,1231,NaN,NaN,NaN,NaN,NaN,1694907,0.00
8,mill_trase_id,VARCHAR,ID-PALM-MILL-00001,ID-PALM-MILL-X,1232,NaN,NaN,NaN,NaN,NaN,1694907,0.00
9,mill_group,VARCHAR,ADIMULIA,ZONERGY,204,NaN,NaN,NaN,NaN,NaN,1694907,0.00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Jumlah baris awal: 1,694,907


## 2. Deteksi missing values

String kosong atau berisi spasi pada kolom teks dinormalisasi menjadi `null`. Ringkasan missing values dihitung dengan agregasi lazy, bukan dengan membuat salinan penuh dataset.

In [3]:
# Tentukan kolom teks dari skema hasil scan.
kolom_teks_awal = [
    name for name, dtype in schema_awal.items()
    if dtype == pl.String
]

# Normalisasi string kosong/spasi menjadi null.
if kolom_teks_awal:
    lf = lf.with_columns([
        pl.when(pl.col(c).str.strip_chars() == "")
          .then(None)
          .otherwise(pl.col(c))
          .alias(c)
        for c in kolom_teks_awal
    ])

def ringkasan_null_lazy(lazy_data, kolom, jumlah_baris):
    if not kolom:
        return pl.DataFrame({"kolom": [], "jumlah_null": [], "persen_null": []})
    hasil = lazy_data.select([
        pl.col(c).null_count().alias(c) for c in kolom
    ]).collect(engine="streaming")
    records = []
    for c in kolom:
        n_null = int(hasil[c][0])
        if n_null > 0:
            records.append({
                "kolom": c,
                "jumlah_null": n_null,
                "persen_null": round(n_null / jumlah_baris * 100, 2) if jumlah_baris else 0.0,
            })
    return pl.DataFrame(records).sort("jumlah_null", descending=True) if records else pl.DataFrame({"kolom": [], "jumlah_null": [], "persen_null": []})

null_summary_awal = ringkasan_null_lazy(lf, kolom_awal, total_baris_awal)
print("Ringkasan missing values sebelum imputasi:")
display(null_summary_awal)
total_null_awal = int(null_summary_awal["jumlah_null"].sum()) if null_summary_awal.height else 0
print(f"Total null yang terdeteksi: {total_null_awal:,}")

Ringkasan missing values sebelum imputasi:


kolom,jumlah_null,persen_null
str,i64,f64
"""total_emission_exposure""",1025290,60.49
"""net_emission_luc_exposure""",1025290,60.49
"""emission_subsidence_exposure""",1025290,60.49
"""gross_emission_luc_exposure""",1025290,60.49
"""emissions_fire_on_peat_exposur…",1025290,60.49
"""fob""",10957,0.65
"""palm_oil_deforestation_10_year…",10233,0.6
"""palm_oil_deforestation_annual_…",10233,0.6
"""oil_palm_ha""",10233,0.6


Total null yang terdeteksi: 5,169,298


## 3. Penanganan missing values

- Kolom ukuran numerik yang memiliki null diisi median.
- Kolom teks yang memiliki null diisi label `Tidak diketahui`.
- Kolom ID, tahun, kode, koordinat, persentase, skor, dan indikator tidak dipilih otomatis sebagai ukuran untuk imputasi median.
- Null pada kolom numerik lain dipertahankan agar tidak membuat asumsi yang keliru.

In [4]:
numeric_types = (pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16,
                 pl.UInt32, pl.UInt64, pl.Float32, pl.Float64)
schema = lf.collect_schema()
numeric_cols = [name for name, dtype in schema.items() if dtype in numeric_types]

def nama_mirip_identitas_atau_kategori(name):
    s = name.lower()
    tokens = ["id", "code", "kode", "year", "tahun", "score", "rating",
              "latitude", "longitude", "lat", "lon", "percent", "percentage",
              "share", "flag", "is_"]
    return any(token in s for token in tokens)

ukuran_keywords = ["volume", "quantity", "tonne", "ton", "production", "produksi",
                   "emission", "deforestation", "area", "distance", "cost", "flow",
                   "trade", "export", "import", "amount", "weight", "ha"]
kolom_ukuran = [
    c for c in numeric_cols
    if not nama_mirip_identitas_atau_kategori(c)
    and any(k in c.lower() for k in ukuran_keywords)
]
print("Kolom numerik:", numeric_cols)
print("Kandidat kolom ukuran:", kolom_ukuran)

# Hitung median semua kandidat dalam satu agregasi, hanya hasil satu baris yang masuk RAM.
kolom_diisi_median = []
median_values = {}
if kolom_ukuran:
    median_row = lf.select([
        pl.col(c).median().alias(c) for c in kolom_ukuran
    ]).collect(engine="streaming")
    for c in kolom_ukuran:
        median_values[c] = median_row[c][0]
        null_count = int(lf.select(pl.col(c).null_count()).collect(engine="streaming")[c][0])
        if null_count > 0 and median_values[c] is not None:
            kolom_diisi_median.append(c)

if kolom_diisi_median:
    lf = lf.with_columns([
        pl.col(c).fill_null(median_values[c]).alias(c)
        for c in kolom_diisi_median
    ])

# Isi null pada kolom teks; hitung dulu kolom yang benar-benar mengandung null.
kolom_diisi_label = []
if kolom_teks_awal:
    null_text_row = lf.select([
        pl.col(c).null_count().alias(c) for c in kolom_teks_awal
    ]).collect(engine="streaming")
    kolom_diisi_label = [c for c in kolom_teks_awal if int(null_text_row[c][0]) > 0]
    if kolom_diisi_label:
        lf = lf.with_columns([
            pl.col(c).fill_null("Tidak diketahui").alias(c)
            for c in kolom_diisi_label
        ])

print("Kolom numerik diisi median:", kolom_diisi_median)
print("Kolom teks diisi label:", kolom_diisi_label)
print("Penanganan missing values selesai (pipeline masih lazy).")

Kolom numerik: ['year', 'volume']
Kandidat kolom ukuran: ['volume']
Kolom numerik diisi median: []
Kolom teks diisi label: ['country_of_first_import_trase_id', 'palm_oil_deforestation_10_year_total_exposure', 'palm_oil_deforestation_annual_exposure', 'total_emission_exposure', 'net_emission_luc_exposure', 'emission_subsidence_exposure', 'gross_emission_luc_exposure', 'emissions_fire_on_peat_exposure', 'oil_palm_ha', 'fob']
Penanganan missing values selesai (pipeline masih lazy).


## 4. Deteksi dan penanganan outlier dengan IQR

Satu kolom numerik yang relevan dipilih berdasarkan nama. Kuartil dihitung sebagai agregasi, lalu nilai ekstrem dibatasi ke rentang IQR (Winsorization), bukan menghapus baris. Bila tidak ada kandidat yang layak, tahap ini dilewati dan dicatat.

In [5]:
KOLOM_OUTLIER = kolom_ukuran[0] if kolom_ukuran else None
df_outlier_handled = lf
outlier_info = {}

if KOLOM_OUTLIER is None:
    print("Tidak ditemukan kolom numerik ukuran yang aman dipilih otomatis; nilai tidak diubah.")
else:
    kuartil = lf.select([
        pl.col(KOLOM_OUTLIER).quantile(0.25).alias("q1"),
        pl.col(KOLOM_OUTLIER).quantile(0.75).alias("q3"),
    ]).collect(engine="streaming")
    q1, q3 = kuartil["q1"][0], kuartil["q3"][0]
    if q1 is None or q3 is None:
        print(f"Kolom {KOLOM_OUTLIER} tidak memiliki data numerik valid; tahap dilewati.")
        KOLOM_OUTLIER = None
    else:
        iqr = q3 - q1
        batas_bawah = q1 - 1.5 * iqr
        batas_atas = q3 + 1.5 * iqr
        jumlah_outlier = int(
            lf.select(
                ((pl.col(KOLOM_OUTLIER) < batas_bawah) |
                 (pl.col(KOLOM_OUTLIER) > batas_atas)).sum().alias("n")
            ).collect(engine="streaming")["n"][0] or 0
        )
        print(f"Kolom diperiksa: {KOLOM_OUTLIER}")
        print(f"Q1={q1}; Q3={q3}; IQR={iqr}")
        print(f"Batas bawah={batas_bawah}; batas atas={batas_atas}")
        print(f"Nilai di luar batas IQR: {jumlah_outlier:,}")
        df_outlier_handled = lf.with_columns(
            pl.col(KOLOM_OUTLIER).clip(lower_bound=batas_bawah, upper_bound=batas_atas).alias(KOLOM_OUTLIER)
        )
        outlier_info = {
            "kolom": KOLOM_OUTLIER, "q1": q1, "q3": q3, "iqr": iqr,
            "batas_bawah": batas_bawah, "batas_atas": batas_atas,
            "jumlah_outlier": jumlah_outlier,
        }

print("Pipeline penanganan outlier siap.")

Kolom diperiksa: volume
Q1=0.1521182186311248; Q3=21.21685539046564; IQR=21.064737171834516
Batas bawah=-31.44498753912065; batas atas=52.813961148217416
Nilai di luar batas IQR: 281,597
Pipeline penanganan outlier siap.


## 5. Deduplikasi baris identik

Tidak menggunakan `mill_trase_id` atau `refinery_trase_id` sebagai kunci tunggal karena satu entitas dapat muncul pada banyak observasi. Data hasil cleaning dari Polars ditulis sementara ke Parquet; DuckDB kemudian menghapus duplikat yang identik di seluruh kolom. Batas memori dan direktori sementara diatur pada bagian persiapan.

In [6]:
# Tulis pipeline Polars ke file sementara secara streaming.
# File sementara memisahkan tahap transformasi dari deduplikasi dan mengurangi kebutuhan RAM.
if TEMP_PATH.exists():
    TEMP_PATH.unlink()
df_outlier_handled.sink_parquet(TEMP_PATH, compression="zstd", maintain_order=False)

jumlah_sebelum_deduplikasi = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{sql_path(TEMP_PATH)}')"
).fetchone()[0]

# DISTINCT menghapus baris yang identik pada semua kolom, tanpa menebak kunci bisnis.
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
con.execute(f"""
    COPY (
        SELECT DISTINCT *
        FROM read_parquet('{sql_path(TEMP_PATH)}')
    ) TO '{sql_path(OUTPUT_PATH)}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

jumlah_setelah_deduplikasi = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{sql_path(OUTPUT_PATH)}')"
).fetchone()[0]
jumlah_duplikat_dihapus = jumlah_sebelum_deduplikasi - jumlah_setelah_deduplikasi

df_dedup = pl.scan_parquet(OUTPUT_PATH)
print(f"Baris sebelum deduplikasi: {jumlah_sebelum_deduplikasi:,}")
print(f"Baris sesudah deduplikasi: {jumlah_setelah_deduplikasi:,}")
print(f"Baris identik yang dihapus: {jumlah_duplikat_dihapus:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baris sebelum deduplikasi: 1,694,907
Baris sesudah deduplikasi: 1,694,907
Baris identik yang dihapus: 0


## 6. Validasi data akhir dengan DuckDB dan Polars

Validasi dilakukan terhadap Parquet final. Agregasi mengembalikan hasil kecil; notebook tidak menjalankan `unique()` kedua kali pada seluruh dataset.

In [7]:
# Validasi jumlah baris dari file final.
validasi_umum = con.execute(f"""
    SELECT COUNT(*) AS total_baris
    FROM read_parquet('{sql_path(OUTPUT_PATH)}')
""").fetchdf()
display(validasi_umum)

# Ringkasan null akhir dihitung melalui scan Parquet.
lf_final = pl.scan_parquet(OUTPUT_PATH)
skema_final = lf_final.collect_schema()
kolom_final = skema_final.names()
null_summary_akhir = ringkasan_null_lazy(lf_final, kolom_final, jumlah_setelah_deduplikasi)
print("Kolom yang masih memiliki null setelah cleaning:")
display(null_summary_akhir)
total_null_akhir = int(null_summary_akhir["jumlah_null"].sum()) if null_summary_akhir.height else 0
print(f"Total null tersisa: {total_null_akhir:,}")

# Validasi rentang tahun, jika kolom tersedia.
kolom_tahun = next((c for c in kolom_final if c.lower() in ["year", "tahun", "trade_year"]), None)
if kolom_tahun:
    tahun_sql = kolom_tahun.replace('"', '""')
    validasi_tahun = con.execute(f"""
        SELECT MIN("{tahun_sql}") AS tahun_min,
               MAX("{tahun_sql}") AS tahun_max,
               COUNT(*) FILTER (WHERE "{tahun_sql}" IS NULL) AS tahun_null
        FROM read_parquet('{sql_path(OUTPUT_PATH)}')
    """).fetchdf()
    display(validasi_tahun)
else:
    print("Kolom tahun tidak terdeteksi; validasi rentang tahun dilewati.")

print("Validasi selesai.")

,total_baris
0,1694907


Kolom yang masih memiliki null setelah cleaning:


kolom,jumlah_null,persen_null
null,null,null


Total null tersisa: 0


,tahun_min,tahun_max,tahun_null
0,2013,2022,0


Validasi selesai.


## 7. Simpan laporan cleaning

Data bersih sudah disimpan sebagai Parquet pada tahap deduplikasi. Sel ini menyimpan ringkasan keputusan dan hasil aktual.

In [8]:
catatan = [
    "MILESTONE 2 — RINGKASAN DATA CLEANING",
    f"File input: {DATA_PATH.name}",
    f"Ukuran CSV (MB): {DATA_PATH.stat().st_size / (1024**2):.2f}",
    f"Dimensi awal: {total_baris_awal:,} baris x {len(kolom_awal)} kolom",
    f"Dimensi akhir: {jumlah_setelah_deduplikasi:,} baris x {len(kolom_final)} kolom",
    f"Kolom teks diisi label: {kolom_diisi_label}",
    f"Kolom numerik diisi median: {kolom_diisi_median}",
    f"Kolom outlier: {KOLOM_OUTLIER}",
    f"Informasi outlier: {outlier_info}",
    f"Baris identik dihapus: {jumlah_duplikat_dihapus:,}",
    f"Total null akhir: {total_null_akhir:,}",
    f"Output Parquet: {OUTPUT_PATH}",
    "Catatan: deduplikasi dilakukan pada baris identik di seluruh kolom; tidak memakai ID entitas sebagai kunci tunggal.",
]
REPORT_PATH.write_text("\n".join(catatan), encoding="utf-8")

print(f"Baris awal       : {total_baris_awal:,}")
print(f"Baris akhir      : {jumlah_setelah_deduplikasi:,}")
print(f"Duplikat dihapus : {jumlah_duplikat_dihapus:,}")
print(f"Parquet          : {OUTPUT_PATH}")
print(f"Laporan          : {REPORT_PATH}")
print("Semua tahapan Milestone 2 selesai dijalankan.")

Baris awal       : 1,694,907
Baris akhir      : 1,694,907
Duplikat dihapus : 0
Parquet          : /home/jovyan/work/data/processed/indonesia_palm_oil_bersih.parquet
Laporan          : /home/jovyan/work/data/processed/ringkasan_cleaning.txt
Semua tahapan Milestone 2 selesai dijalankan.


## Catatan interpretasi hasil

Setelah seluruh sel dijalankan, gunakan angka aktual yang muncul untuk laporan:
- **Profiling:** ringkasan DuckDB `SUMMARIZE` dan jumlah baris awal.
- **Missing values:** kolom dengan null terbanyak sebelum cleaning, serta kolom yang diisi median/label.
- **Outlier:** nama kolom, batas IQR, dan jumlah nilai di luar batas; bila tidak ada kandidat, jelaskan bahwa tahap tidak mengubah nilai.
- **Duplikasi:** jumlah baris identik yang dihapus oleh DuckDB.
- **Validasi:** jumlah baris akhir, null yang tersisa, rentang tahun jika tersedia, serta lokasi file Parquet.

**Catatan:** meskipun versi ini lebih hemat memori, proses `SUMMARIZE`, perhitungan kuartil, dan `DISTINCT` tetap membutuhkan waktu karena harus membaca data besar. Pastikan ruang disk kosong cukup untuk file sementara dan hasil Parquet. Jika notebook sebelumnya pernah memiliki variabel lama, pilih **Restart Kernel** lalu jalankan sel dari atas secara berurutan.